# Домашняя работа 4

## Подготовка

In [ ]:
from pathlib import Path
import json
import hashlib
from copy import deepcopy
import requests
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = DATA
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

## Исходники домашней работы

In [ ]:
manifest = json.loads((LOCAL / "manifest.json").read_text(encoding="utf-8"))
cache = {}
for info in manifest["pages"]:
    cache[info["request_url"]] = json.loads(
        (LOCAL / info["file"]).read_text(encoding="utf-8")
    )
def cached_loader(url):
    return cache[url]
print(manifest["selected_lei"])
print("Сохранённых страниц:", len(cache))

## Готовая функция обхода страниц

In [ ]:
def collect_pages(first_url, loader, max_pages=10):
    url = first_url
    seen_urls = set()
    records = []
    expected_total = None
    for page_number in range(max_pages):
        if url is None:
            break
        if url in seen_urls:
            raise ValueError("Цикл next")
        seen_urls.add(url)
        page = loader(url)
        if not isinstance(page.get("data"), list):
            raise ValueError("data должен быть списком")
        total = page["meta"]["pagination"]["total"]
        if expected_total is None:
            expected_total = total
        if total != expected_total:
            raise ValueError("Число записей изменилось между страницами")
        records.extend(page["data"])
        url = page["links"].get("next")
    if url is not None:
        raise ValueError("Предел страниц: получение не завершено")
    ids = [record["id"] for record in records]
    if len(ids) != len(set(ids)):
        raise ValueError("Повтор ключа между страницами")
    if len(records) != expected_total:
        raise ValueError("Неполный набор")
    return records

## H1 Получение полного справочника

## H2 Пакет исходников и таблица

## H3 Покрытие записей и организаций

## H+ Проверки испорченных копий страниц